In [ ]:
'''
    ### Main Goals of Notebook 03 - Model Architecture and Training

    1. **Introduction & Objectives**  
    Write a short introduction explaining the purpose of this notebook and how it 
    follows Notebook 02 (preprocessing is done, now we build and train the model).

    2. **Load the Data Generators**  
    Import and reload the train and validation generators created in Notebook 02. 
    This ensures the pipeline is connected and ready for training.

    3. **Build the Model Architecture**  
    Choose and create the model: Use Transfer Learning with ResNet50 (freeze base layers + add 
    custom classification head). This is recommended because it gives better 
    performance on medical images.

    4. **Compile the Model**  
    Set optimizer (Adam), loss (binary_crossentropy), and metrics (accuracy + AUC). 
    This prepares the model for effective training.

    5. **Add Training Callbacks**  
    Include ModelCheckpoint, EarlyStopping, and ReduceLROnPlateau. 
    These tools control training, prevent overfitting, and save the best model automatically.

    6. **Train the Model**  
    Train using the train generator, validation generator, and class weights (to handle imbalance). 
    Run for a reasonable number of epochs.

    7. **Visualize Training Results**  
    Plot accuracy, loss, and AUC curves for both training and validation sets. 
    This helps analyze if the model is learning well or overfitting.

    8. **Save the Model & Conclusion**  
    Save the best model file. Write a short conclusion with key choices, training results, 
    and next steps (evaluation + Grad-CAM in Notebook 04).
'''

In [ ]:
# 1 Introduction & Objectives
'''
    After completing the Exploratory Data Analysis in Notebook 01 and building a robust preprocessing 
    pipeline with data augmentation and class weights in Notebook 02, we now move to the core 
    phase of the project: model construction and training

    Notebook 01 revealed a significant class imbalance in the Chest X-Ray dataset (25.71% Normal vs 
                                                                                74.29% Pneumonia), 
    while Notebook 02 addressed this challenge through targeted data augmentation and calculated 
    class weights. The preprocessing pipeline is now fully validated, with images 
    resized to 224×224 and properly normalized.

    ## Objectives

    The main objectives of this notebook are:

    1. Design and implement a Deep Learning model using Transfer Learning (ResNet50) 
        adapted to the medical imaging task.
    2. Connect the model to the preprocessed data generators from Notebook 02.
    3. Train the model efficiently while handling class imbalance using the computed class weights.
    4. Monitor the training process with appropriate callbacks and visualize learning curves.
    5. Save the best performing model as a key deliverable for the PFA project.

    This notebook represents the modeling and training stage of our intelligent system for 
    automatic pneumonia detection from chest X-rays, as defined in the Cahier des Charges.
'''



In [5]:
# 2 Loading the Data Generators

import tensorflow as tf
import albumentations
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

DATA_DIR = "../data/raw/chest_xray"


train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,              
    rotation_range=15,           
    width_shift_range=0.1,       
    height_shift_range=0.1,      
    zoom_range=0.12,             
    horizontal_flip=True,        
    fill_mode='nearest'          
)

# 2 Création des générateurs de données + Redimensionnement à 224x224
train_generator = train_datagen.flow_from_directory(
    directory=DATA_DIR + "/train",      
    target_size=(224, 224),             
    batch_size=32,
    class_mode='binary',                
    shuffle=True                       
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

val_generator = val_datagen.flow_from_directory(
    directory=DATA_DIR + "/val",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

class_weight = {
    0:1.9448,
    1:0.6730
}

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.


In [6]:
# 3 Building the Model Architecture

# Using Transfer Learning with ResNet50 (freeze base layers + custom classification head)

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# 1. Load ResNet50 base model (without top layers)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,     
    input_shape=(224, 224, 3)
)

# 2. Freeze the base layers
base_model.trainable = False

# 3. Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)        # Reduce spatial dimensions
x = Dropout(0.5)(x)                    # Regularization to prevent overfitting
x = Dense(128, activation='relu')(x)   # Small fully connected layer
output = Dense(1, activation='sigmoid')(x)   # Binary classification (Normal / Pneumonia)

# 4. Create the final model
model = Model(inputs=base_model.input, outputs=output)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [7]:
# 4 Module Compile Time
# optimizer (Adam), loss (binary_crossentropy), and metrics (accuracy + AUC). 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        AUC(name="auc")
    ]
)


In [8]:
# 5 Training Callbacks
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint(
        filepath="best_model.h5",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [10]:
# 6 Module Train

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks
)

Epoch 1/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 781ms/step - accuracy: 0.5291 - auc: 0.5044 - loss: 0.7345
Epoch 1: val_loss improved from None to 0.70082, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 132s 792ms/step - accuracy: 0.5111 - auc: 0.5189 - loss: 0.7146 - val_accuracy: 0.5000 - val_auc: 0.7734 - val_loss: 0.7008 - learning_rate: 1.0000e-04
Epoch 2/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 477ms/step - accuracy: 0.5544 - auc: 0.5692 - loss: 0.6856
Epoch 2: val_loss improved from 0.70082 to 0.68874, saving model to best_model.h5



Epoch 2: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 79s 481ms/step - accuracy: 0.5686 - auc: 0.5956 - loss: 0.6799 - val_accuracy: 0.5000 - val_auc: 0.8281 - val_loss: 0.6887 - learning_rate: 1.0000e-04
Epoch 3/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step - accuracy: 0.5706 - auc: 0.6259 - loss: 0.6680
Epoch 3: val_loss improved from 0.68874 to 0.65556, saving model to best_model.h5



Epoch 3: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 73s 448ms/step - accuracy: 0.5801 - auc: 0.6355 - loss: 0.6651 - val_accuracy: 0.8125 - val_auc: 0.8516 - val_loss: 0.6556 - learning_rate: 1.0000e-04
Epoch 4/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.6531 - auc: 0.6963 - loss: 0.6411
Epoch 4: val_loss improved from 0.65556 to 0.64476, saving model to best_model.h5



Epoch 4: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 77s 468ms/step - accuracy: 0.6388 - auc: 0.6975 - loss: 0.6426 - val_accuracy: 0.8125 - val_auc: 0.8516 - val_loss: 0.6448 - learning_rate: 1.0000e-04
Epoch 5/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step - accuracy: 0.6635 - auc: 0.7023 - loss: 0.6415
Epoch 5: val_loss did not improve from 0.64476
163/163 ━━━━━━━━━━━━━━━━━━━━ 81s 498ms/step - accuracy: 0.6639 - auc: 0.7167 - loss: 0.6331 - val_accuracy: 0.5000 - val_auc: 0.8516 - val_loss: 0.6523 - learning_rate: 1.0000e-04
Epoch 6/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 560ms/step - accuracy: 0.6827 - auc: 0.7514 - loss: 0.6204
Epoch 6: val_loss improved from 0.64476 to 0.64017, saving model to best_model.h5


...


Epoch 13: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 79s 481ms/step - accuracy: 0.7492 - auc: 0.8175 - loss: 0.5472 - val_accuracy: 0.8125 - val_auc: 0.8594 - val_loss: 0.5733 - learning_rate: 1.0000e-04
Epoch 14/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.7622 - auc: 0.8322 - loss: 0.5371
Epoch 14: val_loss improved from 0.57326 to 0.56562, saving model to best_model.h5



Epoch 14: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 80s 488ms/step - accuracy: 0.7544 - auc: 0.8214 - loss: 0.5407 - val_accuracy: 0.8125 - val_auc: 0.8594 - val_loss: 0.5656 - learning_rate: 1.0000e-04
Epoch 15/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step - accuracy: 0.7712 - auc: 0.8342 - loss: 0.5271
Epoch 15: val_loss improved from 0.56562 to 0.55435, saving model to best_model.h5



Epoch 15: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 80s 486ms/step - accuracy: 0.7674 - auc: 0.8311 - loss: 0.5308 - val_accuracy: 0.7500 - val_auc: 0.8594 - val_loss: 0.5543 - learning_rate: 1.0000e-04
Epoch 16/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 530ms/step - accuracy: 0.7544 - auc: 0.8208 - loss: 0.5382
Epoch 16: val_loss did not improve from 0.55435
163/163 ━━━━━━━━━━━━━━━━━━━━ 87s 533ms/step - accuracy: 0.7621 - auc: 0.8272 - loss: 0.5315 - val_accuracy: 0.7500 - val_auc: 0.8516 - val_loss: 0.5773 - learning_rate: 1.0000e-04
Epoch 17/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step - accuracy: 0.7514 - auc: 0.8280 - loss: 0.5271
Epoch 17: val_loss did not improve from 0.55435
163/163 ━━━━━━━━━━━━━━━━━━━━ 82s 498ms/step - accuracy: 0.7559 - auc: 0.8272 - loss: 0.5295 - val_accuracy: 0.8125 - val_auc: 0.8594 - val_loss: 0.5556 - learning_rate: 1.0000e-04
Epoch 18/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step - accuracy: 0.7676 - auc: 0.8425 - loss: 0.5144
Epoch


Epoch 18: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 80s 491ms/step - accuracy: 0.7671 - auc: 0.8383 - loss: 0.5184 - val_accuracy: 0.7500 - val_auc: 0.8594 - val_loss: 0.5443 - learning_rate: 1.0000e-04
Epoch 19/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step - accuracy: 0.7634 - auc: 0.8334 - loss: 0.5204
Epoch 19: val_loss did not improve from 0.54427
163/163 ━━━━━━━━━━━━━━━━━━━━ 79s 484ms/step - accuracy: 0.7596 - auc: 0.8321 - loss: 0.5205 - val_accuracy: 0.8125 - val_auc: 0.8594 - val_loss: 0.5527 - learning_rate: 1.0000e-04
Epoch 20/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 514ms/step - accuracy: 0.7695 - auc: 0.8368 - loss: 0.5135
Epoch 20: val_loss improved from 0.54427 to 0.53978, saving model to best_model.h5



Epoch 20: finished saving model to best_model.h5
163/163 ━━━━━━━━━━━━━━━━━━━━ 85s 518ms/step - accuracy: 0.7728 - auc: 0.8400 - loss: 0.5145 - val_accuracy: 0.7500 - val_auc: 0.8516 - val_loss: 0.5398 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 20.


In [ ]:
# Visualize Training Results

from tensorflow.keras.models import load_model

model = load_model("best_model.h5")